### Imports

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import scipy.signal
from pathlib import Path
from itertools import count
from string import ascii_lowercase as alc
from scipy import stats, optimize

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize, TwoSlopeNorm, ListedColormap
from matplotlib.offsetbox import AnchoredText
from matplotlib.collections import LineCollection
from matplotlib.ticker import AutoMinorLocator
import seaborn as sns
import cartopy.crs as ccrs
import cmcrameri as scm

# Needs to be added to the path for the imports below to work
import sys
sys.path.append(str(Path(__file__).parent))
import util

In [ ]:
sns.set_theme(rc={'figure.figsize':(8,11)})
sns.set_style("ticks")
# sns.set_context("notebook", font_scale=1, rc={"lines.linewidth": 1.5, "fontname":'Inter'})
plt.rcParams["font.sans-serif"] = ["Inter"]

In [ ]:
# To modify with your data folder. !This does not include GCM outputs.
home_folder = str(Path.home())
script_folder = str(Path.cwd())

data_folder = f"{script_folder}/data"
ice_sheet_folder = f"{data_folder}/ice_sheet_reconstructions"
output_folder = f"{script_folder}/outputs"

# Pattern of North Atlantic convection preconditions AMOC for non-linear responses to meltwater

In [ ]:
database = {'tecac':f"{data_folder}/ice6g_hs1/lgm_control", 
            'xoupa':f"{data_folder}/database/xoupa", 
            'xpfjb':f"{data_folder}/database/xpfjb",
            'xpfjc':f"{data_folder}/database/xpfjc", 
            'xpfjd':f"{data_folder}/database/xpfjd", 
            'xpfje':f"{data_folder}/database/xpfje",
            'xqctb':f"{data_folder}/database/xqctb",
            'xqctc':f"{data_folder}/database/xqctc"}

In [ ]:
color_expt = {'xoupa':'#232E40', 'tecac':'#4A222E',
              'xpfjb':'#8C2E2E', 'xpfjc':'#AD5956',
              'xpfjd':'#004266', 'xpfje':'#6BABC1', 
              'xqctb':'xkcd:ocean green', 'xqctc':"#B48A18"}

start_expt = {'tecac':2552+22000, 'xoupa':22000, 'xpfjb':100000, 'xpfjc':100000, 'xpfjd':100000, 'xpfje':100000, 
              'xqctb':100000, 'xqctc':100000}

name_expt = {'tecac':"ICE6G_noMw", 'xoupa':"GLAC1D_noMw",
              'xpfjb':"ICE6G_mw", 'xpfjc':"ICE6G_GLAC1DMw",
              'xpfjd':"GLAC1D_mw", 'xpfje':"GLAC1D_ICE6GMw",
              'xqctc':"GLAC1D_ts"}

ice_expt = {'tecac':"ICE6G", 'xoupa':"GLAC-1D",
              'xpfjb':"ICE6G", 'xpfjc':"ICE6G",
              'xpfjd':"GLAC-1D", 'xpfje':"GLAC-1D",
            'xqctb':"GLAC-1D", 'xqctc':"GLAC-1D"}

label_expt = {'tecac':"ICE6G control", 'xoupa':"GLAC-1D control",
              'xpfjb':"ICE6G ice-sheet, ICE6G meltwater", 'xpfjc':"ICE6G ice-sheet, GLAC-1D meltwater",
              'xpfjd':"GLAC-1D ice-sheet, GLAC-1D meltwater", 'xpfje':"GLAC-1D ice-sheet, ICE6G meltwater",
              'xqctb':"GLAC-1D ghg transient", 'xqctc':"GLAC-1D full transient"}

In [ ]:
color_fluxes = {'elwg': 'xkcd:deep blue', 'gin':'xkcd:dark pink', 'med':'xkcd:avocado green',
                'arc':'xkcd:kelley green', 'so':'xkcd:lightblue', 'pac':'xkcd:sandy', 'tot':'black'}
label_fluxes = {'elwg':'East Laurentide and\nWest Greenland', 'gin':'GIN seas', 'med':'Mediterranean Sea',
                'arc':'Arctic', 'so':'Southern Ocean', 'pac':'Pacific', 'tot':'Total'}

In [ ]:
color_spans = {'cold':'#2C4251', 
               'warming':'#FFABAB',
               'merid':'#C9381B', 
               'zonal':'#F59F00', 
               'cooling':'#B7DDF6'}

In [ ]:
ds_basin = xr.open_dataset(f"{data_folder}/basin_hadcm3_glac1d_lgm.nc")

masks = {}

masks['arc'] = ds_basin.arctic
masks['na'] = ds_basin.atlantic.sel(latitude=slice(44,79))
masks['tpa'] = ds_basin.atlantic.sel(latitude=slice(-33,44))
masks['sa'] = ds_basin.atlantic.sel(latitude=slice(-60,-33))
masks['pac'] = ds_basin.pacific
masks['so'] = ds_basin.southern
masks['idn'] = ds_basin.indian

color_zones = {'arc':'#A4D4D2',
               'na':'#00316A', 'ena':'xkcd:blue green', 'wna':'xkcd:navy', 
               'tpa':'#306B34', 
               'sa':'#FEE000', 
               'pac':'#B41001', 'npc':'xkcd:light red', 'spc':'xkcd:dark red',
               'idn':'#F68307', 
               'soa':'xkcd:wisteria', 'soi':'xkcd:pinky purple', 'sop':'xkcd:royal purple', 
               'so':'#953878', 
               'tot':'xkcd:black', 
               'gin':'#DFBA47', 'irm':'#CFABA0', 'ls':'#CC573D', 
               'spg':'#C4BE81', 'eur':'#9EA550'}


label_zones = {'arc':'Arctic', 
               'ena':'East North Atlantic', 'wna':'West North Atlantic', 
               'na': 'North Atlantic', 'tpa':'Subtropical Atlantic', 'sa':'South Atlantic',
               'pac':'Pacific', 'npc':'North Pacific', 'spc':'South Pacific',
               'idn':'Indian',
               'so':'Southern', 'soa':'SO Atlantic', 'soi':'SO Indian', 'sop':'SO Pacific', 
               'tot':'Total',
               'gin':'GIN seas', 'irm':'Irminger Sea', 'ls':'Labrador Sea', 
               'spg':'Subpolar gyre', 'eur':'European coast'}


color_zones['nor'] = color_zones['gin']
label_zones['nor'] = "Norwegian Sea"
color_zones['icd'] = color_zones['eur']
label_zones['icd'] = "Iceland Basin"

masks_na = {}

masks_na['gin'] = xr.concat([ds_basin.atlantic.sel(longitude=slice(340,360)).sel(latitude=slice(64,79)),
                          ds_basin.atlantic.sel(longitude=slice(0,20)).sel(latitude=slice(64,79))], 
                         dim='longitude')
masks_na['eur'] = xr.concat([ds_basin.atlantic.sel(longitude=slice(340,360)).sel(latitude=slice(44,64)),
                          ds_basin.atlantic.sel(longitude=slice(0,2)).sel(latitude=slice(44,64))], 
                         dim='longitude')
masks_na['irm'] = ds_basin.atlantic.sel(longitude=slice(316,339)).sel(latitude=slice(53,70))
masks_na['ls'] = ds_basin.atlantic.sel(longitude=slice(280,315)).sel(latitude=slice(53,79))
masks_na['spg'] = ds_basin.atlantic.sel(longitude=slice(280,339)).sel(latitude=slice(44,53))
masks_na['arc'] = ds_basin.arctic


## Figure 1 - AMOC response to deglacial meltwater

Orbital parameters produced using https://vo.imcce.fr/insola/earth/online/earth/online/index.php and verified with https://biocycle.atmos.colostate.edu/shiny/Milankovitch/.

In [ ]:
amoc, amocn = {}, {}

for expt in database.keys():
    print(f"Loading {expt}...")
    amoc[expt] = xr.open_dataset(
        f"{database[expt]}/time_series/{expt}.merid.annual.nc").Merid_Atlantic.sel(
        latitude=26.5, method='nearest').max('depth')
    amocn[expt] = amoc[expt] - np.mean(amoc[expt])

In [ ]:
fluxes, lsm = {}, {}
time = {}

In [ ]:
# ICE6G

ds_discharge = xr.open_dataset(f"{data_folder}/mw_inputs/xpfj.wfix.ice6g_ts.shift.nc",decode_times=False)
ds_discharge.attrs['waterfix'] = 'GLAC-1D'
ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/ice6g.omask.nc")
ds_wfix = xr.open_dataset(f"{data_folder}/lgm_inputs/teadv3.qrparm.waterfix.nc")

fluxes['ICE6G'] = plotting.create_discharge_ts(plotting.remove_waterfix(
    saving.ancil_to_discharge(ds_discharge), ds_wfix),
                                    ds_lsm, details='low', rmean=1)

lsm['ICE6G'] = ds_lsm.lsm

time['ICE6G'] = ds_discharge.t.values - 100000

In [ ]:
# GLAC-1D

ds_discharge = xr.open_dataset(f"{data_folder}/mw_inputs/xoup.wfix.glac_ts.nc",decode_times=False)
ds_discharge.attrs['waterfix'] = 'GLAC-1D'

ds_wfix = xr.open_dataset(f"{data_folder}/lgm_inputs/qrparm.waterfix.hadcm3.nc")
ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/temev.qrparm.omask.nc")

fluxes['GLAC-1D'] = util.create_discharge_ts(util.remove_waterfix(
    util.ancil_to_discharge(ds_discharge), ds_wfix),
                                    ds_lsm, details='low', rmean=1)
lsm['GLAC-1D'] = ds_lsm.lsm

time['GLAC-1D'] = ds_discharge.t.values

In [ ]:
flux_dict, flux_dict_tot, flux_dict_interp = {}, {}, {}
for mw in ['GLAC-1D', 'ICE6G']:
    flux_dict[mw], flux_dict_tot[mw], flux_dict_interp[mw] = {}, {}, {}
    for key in fluxes[mw].keys():
        if key!='tot':
            flux_dict[mw][key] = fluxes[mw][key].value
        flux_dict_tot[mw][key] = fluxes[mw][key].value
        flux_dict_interp[mw][key] = np.interp(amoc['xpfjb'].t.dt.year.values-100_000, time[mw], flux_dict_tot[mw][key]) - np.mean(flux_dict_tot[mw][key])

expt_mw = {'GLAC-1D':['xoupa', 'tecac', 'xpfjd', 'xpfjc', 'xqctc'],
            'ICE6G':['xoupa', 'tecac', 'xpfjb', 'xpfje']}


In [ ]:
co2_ts = pd.read_csv(f"{data_folder}/co2_deglac.csv")

In [ ]:
orbital_data = np.array([]).reshape(0, 2)  # Initialize an empty array with 2 columns
with open(f"{data_folder}/imcce.txt", 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            kya = float(parts[0])
            insolation = float(parts[1])
            orbital_data = np.append(orbital_data, np.array([[kya*1000, insolation]]), axis=0)

# After reading, convert to pandas DataFrame
orbital_ts = pd.DataFrame(orbital_data, columns=['Years', 'Insolation'])

In [ ]:
fig = plt.figure(figsize=(10,16), dpi=300)

gs = gridspec.GridSpec(nrows=5, ncols=1, hspace=0, wspace=0.15, height_ratios=[6,6,8,6,8])
axMw, axAMOC = {}, {} 

axCO2 = fig.add_subplot(gs[0, 0])
axMw['GLAC-1D'] = fig.add_subplot(gs[1, 0], sharex=axCO2)
axSOL = axCO2.twinx()
axAMOC['GLAC-1D'] = fig.add_subplot(gs[2, 0], sharex=axMw['GLAC-1D'])
axMw['ICE6G'] = fig.add_subplot(gs[3, 0], sharex=axMw['GLAC-1D'])
axAMOC['ICE6G'] = fig.add_subplot(gs[4, 0], sharex=axMw['GLAC-1D'])

# -----

# C02
# color = '#335228'
color = '#152110'
axCO2.plot(-co2_ts.Years/1000, co2_ts.CO2*1e6, color=color)

axCO2.set_ylabel("CO2 concentrations\n($ppm$)", color=color, size='large')
axCO2.tick_params(axis='y', labelcolor=color)
axCO2.set_ylim([270,390])
for loc in ['right', 'bottom']: axCO2.spines[loc].set_visible(False)
axCO2.xaxis.set_ticks_position('top')
axCO2.xaxis.set_label_position('top')
axCO2.set_xlabel("Years ($ka\,BP$)", size='large') 
axCO2.set_xlim([22,13])

# Insolation
# color = '#97411C'
color = '#6C3D00'
axSOL.plot(-orbital_ts.Years/1000, orbital_ts.Insolation, color=color)
axSOL.set_ylabel("Summer insolation at 65$^{\circ}$N\n($W.m^{-2}$)", color=color, size='large')
axSOL.tick_params(axis='x', colors='None', which='both')
axSOL.tick_params(axis='y', labelcolor=color)
for loc in ['left', 'top', 'bottom']: axSOL.spines[loc].set_visible(False)


# GLAC-1D
mw='GLAC-1D'

# Meltwater
axMw[mw].stackplot(-time[mw]/1000, flux_dict[mw].values(), 
                          labels=label_fluxes.values(), colors=color_fluxes.values(),
                          linewidth=0, alpha=1, zorder=2)

axMw[mw].set_ylabel("Meltwater flux ($Sv$)", size='large')
axMw[mw].legend(title="GLAC-1D meltwater", title_fontproperties={'style':'italic', 'size':'small'},
                       loc="upper left", fontsize='small', edgecolor='white', facecolor='white')
axMw[mw].tick_params(axis='x', colors='None', which='both')
axMw[mw].set_ylim([0,0.37])
axMw[mw].yaxis.set_ticks_position('left')
for loc in ['right', 'top', 'bottom']: axMw[mw].spines[loc].set_visible(False)

# AMOC
for expt in expt_mw[mw][::-1]:
    axAMOC[mw].plot(-(amoc[expt].t.dt.year-start_expt[expt])/1000, util.rmean(amoc[expt].values, 30), 
                    color=color_expt[expt], linestyle="-", 
                    label=f"{name_expt[expt]}")
    axAMOC[mw].plot(-(amoc[expt].t.dt.year-start_expt[expt])/1000, amoc[expt].values, 
                    color=color_expt[expt], linestyle="-", alpha=0.2)

axAMOC[mw].yaxis.set_ticks_position('right')
axAMOC[mw].yaxis.set_label_position('right')
axAMOC[mw].tick_params(axis='x', colors='None', which='both')
for loc in ['left', 'top']: axAMOC[mw].spines[loc].set_visible(False)
axAMOC[mw].set_ylabel("Max AMOC at 26.5°N ($Sv$)", size='large')
axAMOC[mw].legend(loc="upper left", fontsize='small', edgecolor='None', facecolor='None')

# landmarks
landmarks = [20.75, 19.4, 18.2, 17.1, 16.2, 15.2, 14.4, 13.4]
for landmark in landmarks:
    axMw[mw].axvline(landmark, color="xkcd:dark grey", linestyle="--", linewidth=0.5, zorder=1)
    axCO2.axvline(landmark, color="xkcd:dark grey", linestyle="--", linewidth=0.5, zorder=1)
    axAMOC[mw].axvline(landmark, color="xkcd:dark grey", linestyle="--", linewidth=0.5, zorder=1)


# ICE6G
mw='ICE6G'

# Meltwater
axMw[mw].stackplot(-time[mw]/1000, flux_dict[mw].values(), 
                          labels=label_fluxes.values(), colors=color_fluxes.values(),
                          linewidth=0, alpha=1)

axMw[mw].set_ylabel("Meltwater flux ($Sv$)", size='large')
leg = axMw[mw].legend(title="ICE6G meltwater", title_fontproperties={'style':'italic', 'size':'small'},
                       loc="upper left", fontsize='small', edgecolor='white', facecolor='white')
axMw[mw].tick_params(axis='x', colors='None', which='both')
axMw[mw].set_ylim([0,0.37])
axMw[mw].yaxis.set_ticks_position('left')
for loc in ['right', 'top', 'bottom']: axMw[mw].spines[loc].set_visible(False)

# AMOC
for expt in expt_mw[mw][::-1]:
    axAMOC[mw].plot(-(amoc[expt].t.dt.year-start_expt[expt])/1000, util.rmean(amoc[expt].values, 30), 
                    color=color_expt[expt], linestyle="-", 
                    label=f"{name_expt[expt]}")
    axAMOC[mw].plot(-(amoc[expt].t.dt.year-start_expt[expt])/1000, amoc[expt].values, 
                    color=color_expt[expt], linestyle="-", alpha=0.2)

axAMOC[mw].yaxis.set_ticks_position('right')
axAMOC[mw].yaxis.set_label_position('right')
axAMOC[mw].set_xlabel("Years ($ka\,BP$)", size='large')
for loc in ['left', 'top']: axAMOC[mw].spines[loc].set_visible(False)

axAMOC[mw].set_ylabel("Max AMOC at 26.5°N ($Sv$)", size='large')
axAMOC[mw].legend(loc="lower left", fontsize='small', edgecolor='None', facecolor='None')

# -----

# Boxes
for mw in ['ICE6G','GLAC-1D']:
    for loc in ['top','right']: 
        axMw[mw].spines[loc].set_visible(True)
        axMw[mw].spines[loc].set(linewidth=0.25)
    axAMOC[mw].spines['left'].set_visible(True)
    axAMOC[mw].spines['left'].set(linewidth=0.25)


# Other parameters
for ax in [axMw['ICE6G'], axMw['GLAC-1D'], axCO2, axSOL, axAMOC['ICE6G'], axAMOC['GLAC-1D']]:
    ax.minorticks_on()
    ax.tick_params(axis='both', which='minor', size=3)
    ax.tick_params(axis='both', which='major', labelsize='medium')
    ax.grid(which = "both", color='lightgrey', linestyle='-', linewidth=0.2)

# Annotations
axCO2.annotate('a', xy=(0.02,0.9), xycoords='axes fraction', size=14, fontweight='bold')
axMw['GLAC-1D'].annotate('b', xy=(0.96,0.9), xycoords='axes fraction', size=14, fontweight='bold')
axAMOC['GLAC-1D'].annotate('c', xy=(0.96,0.9), xycoords='axes fraction', size=14, fontweight='bold')
axMw['ICE6G'].annotate('d', xy=(0.96,0.9), xycoords='axes fraction', size=14, fontweight='bold')
axAMOC['ICE6G'].annotate('e', xy=(0.02,0.9), xycoords='axes fraction', size=14, fontweight='bold')


In [ ]:
# fig.savefig(f"{output_folder}/ts.png", bbox_extra_artists=(), bbox_inches='tight', format='png')

## Figure 2 - Ice-sheet reconstructions and atmospheric circulation

In [ ]:
lsm = {}

ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/ice6g.omask.nc")
lsm['ICE6G'] = ds_lsm.lsm

ds_lsm = xr.open_dataset(f"{data_folder}/lgm_inputs/temev.qrparm.omask.nc")
lsm['GLAC-1D'] = ds_lsm.lsm


In [ ]:
means = {}

means['mld'], means['sst'] = {}, {}

# MLD
means['mld'] = {}
means['mld']['tecac'] = xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.mld.winter.summary.nc").field653
means['mld']['xoupa'] = xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.mld.winter.summary.nc").mixLyrDpth_mm_uo

# Sea ice
means['wice'], means['sice'] = {}, {}
means['wice']['tecac'] = xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.wice.annual.summary.nc").iceconc
means['sice']['tecac'] = xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.sice.annual.summary.nc").iceconc
means['wice']['xoupa'] = xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.wice.annual.summary.nc").iceconc_mm_srf
means['sice']['xoupa'] = xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.sice.annual.summary.nc").iceconc_mm_srf

# Mean sea level pressure
means['mslp'] = {}
means['mslp']['tecac'] = util.extend_lon(
    xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.mslp.annual.summary.nc").p, 1, 'longitude')/100
means['mslp']['xoupa'] = util.extend_lon(
    xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.mslp.annual.summary.nc").p_mm_msl, 1, 'longitude')/100

# Wind stress
means['u'], means['v'], means['ws'] = {}, {}, {}
means['u']['tecac'] = xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.u.annual.summary.nc").u
means['v']['tecac'] = xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.v.annual.summary.nc").v
means['ws']['tecac'] = xr.open_dataset(f"{database['tecac']}/control_summaries/{'tecac'}.ws.annual.summary.nc").ws
means['u']['xoupa'] = xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.u.annual.summary.nc").u_mm_10m
means['v']['xoupa'] = xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.v.annual.summary.nc").v_mm_10m
means['ws']['xoupa'] = xr.open_dataset(f"{database['xoupa']}/control_summaries/{'xoupa'}.ws.annual.summary.nc").ws



In [ ]:
# Ice Sheets
print("__ Processing Ice sheets")

means['ice'] = {}

ds = xr.open_dataset(f"{ice_sheet_folder}/GLAC1DHiceF26.nc")
means['ice']['xoupa'] = xr.where(ds.HGLOBH.sel(T122KP1=-21.)<10, np.nan, ds.HGLOBH.sel(T122KP1=-21.)).rename(
    {'XLONGLOBP5':'longitude', 'YLATGLOBP25':'latitude'})
lon_processed = np.where(means['ice']['xoupa'].longitude>360, means['ice']['xoupa'].longitude-360, means['ice']['xoupa'].longitude)
means['ice']['xoupa'] = means['ice']['xoupa'].assign_coords(longitude=lon_processed).sortby('longitude')

ds = xr.open_dataset(f"{ice_sheet_folder}/I6_C.VM5a_10min.21.nc").rename(
    {'lon':'longitude', 'lat':'latitude'})
means['ice']['tecac'] = xr.where(ds.sftgif==0, np.nan, ds.sftgif*ds.Orog)/100
means['ice']['tecac'] = means['ice']['tecac'].interp(longitude=means['ice']['xoupa'].longitude).interp(latitude=means['ice']['xoupa'].latitude)

In [ ]:
masks_na['gin'] = xr.concat([ds_basin.atlantic.sel(longitude=slice(340,360)).sel(latitude=slice(64,79)),
                          ds_basin.atlantic.sel(longitude=slice(0,20)).sel(latitude=slice(64,79))], 
                         dim='longitude')
masks_na['eur'] = xr.concat([ds_basin.atlantic.sel(longitude=slice(340,360)).sel(latitude=slice(44,64)),
                          ds_basin.atlantic.sel(longitude=slice(0,2)).sel(latitude=slice(44,64))], 
                         dim='longitude')
masks_na['irm'] = ds_basin.atlantic.sel(longitude=slice(316,339)).sel(latitude=slice(53,70))
masks_na['ls'] = ds_basin.atlantic.sel(longitude=slice(280,315)).sel(latitude=slice(53,79))
masks_na['spg'] = ds_basin.atlantic.sel(longitude=slice(280,339)).sel(latitude=slice(44,53))
masks_na['arc'] = ds_basin.arctic


In [ ]:
dwfs = {}

dwfs['nor'] = util.cycle_box(0, 20, 60, 79)
dwfs['icd'] = util.cycle_box(339, 355, 53, 65)
dwfs['irm'] = util.cycle_box(316, 339, 53, 70)
dwfs['ls'] = util.cycle_box(280, 315, 53, 79)

color_zones['nor'] = color_zones['gin']
label_zones['nor'] = "Norwegian Sea"
color_zones['icd'] = color_zones['eur']
label_zones['icd'] = "Iceland Basin"

In [ ]:
# High and low pressure systems

pressure_systems = {'arc':{}, 'icd':{}, 'az':{}}
marker_pressure = {'arc':'D', 'icd':'P', 'az':'o'}
label_pressure = {'arc':'Polar High', 'icd':'Icelandic Low', 'az':'Azores High'}

for expt in ['tecac', 'xoupa']:
    temp = means['mslp'][expt].sortby('latitude').sel(latitude=slice(70,90))
    pressure_systems['arc'][expt] = (temp.longitude.isel(longitude=temp.argmax(dim=['latitude','longitude'])['longitude']),
                                        temp.latitude.isel(latitude=temp.argmax(dim=['latitude','longitude'])['latitude']))

    temp = means['mslp'][expt].sortby('latitude').sel(latitude=slice(50,70)).sel(longitude=slice(300,360))
    pressure_systems['icd'][expt] = (temp.longitude.isel(longitude=temp.argmin(dim=['latitude','longitude'])['longitude']),
                                     temp.latitude.isel(latitude=temp.argmin(dim=['latitude','longitude'])['latitude']))

    temp = means['mslp'][expt].sortby('latitude').sel(latitude=slice(20,50)).sel(longitude=slice(280,360))
    pressure_systems['az'][expt] = (temp.longitude.isel(longitude=temp.argmax(dim=['latitude','longitude'])['longitude']),
                                    temp.latitude.isel(latitude=temp.argmax(dim=['latitude','longitude'])['latitude']))


In [ ]:
bands = {}

bands['arc'] = [(0,360), (80,90)]
bands['high_na'] = [(335,380), (55,80)]
bands['low_na'] = [(335,380), (35,50)]

label_band = {'arc':'Arctic (>80° N)', 'high_na':"North Atlantic (55° N-80° N)", 'low_na':"North Atlantic (35° N-50° N)"}

In [ ]:
cmaps = {'mld':'Greens', 'mld_diff':scm.cm.cork,
         'ice': 'Blues_r', 'ice_diff':scm.cm.vik.reversed(),
         'ws':matplotlib.colors.ListedColormap(scm.cm.broc(np.linspace(0.5, 1, 128))), 'ws_diff':scm.cm.broc}

norms = {'mld': Normalize(vmin=0, vmax=600), 'mld_diff':TwoSlopeNorm(vmin=-400, vcenter=0, vmax=400),
         'ice': Normalize(vmin=0, vmax=4000), 'ice_diff':TwoSlopeNorm(vmin=-3000, vcenter=0, vmax=3000),
         'ws': Normalize(vmin=0, vmax=10), 'ws_diff': TwoSlopeNorm(vmin=-5, vcenter=0, vmax=5)}

In [ ]:
projection_map = ccrs.NearsidePerspective(central_longitude=-30, central_latitude=70, satellite_height=8000000)
alpha_hist = 0.05
nbins = 50

axICE6G, axGLAC1D, axDIFF = {}, {}, {}

fig = plt.figure(figsize=(15, 12), dpi=300)
axHIST = {}

gs = fig.add_gridspec(
    nrows=3, ncols=1,
    height_ratios=[1, 1, 1],
    wspace=0.2, hspace=0.1
)

for row, var in enumerate(['ice', 'mld', 'ws']):
    subgs = gs[row].subgridspec(1, 3, wspace=0.05)
    axICE6G[var] = fig.add_subplot(subgs[0, 0], projection=projection_map)
    axGLAC1D[var] = fig.add_subplot(subgs[0, 1], projection=projection_map)
    axDIFF[var] = fig.add_subplot(subgs[0, 2], projection=projection_map)

# -----

# Plot variables
labels_cbar = {'mld':"Winter mixed layer depth\n($m$)", 'ice':"Ice thickness\n($m$)", 'ws':"Wind stress\n($m.s^{-1}$)"}

for var in ['mld', 'ice', 'ws']:        
    axICE6G[var].pcolormesh(means[var]['tecac'].longitude, means[var]['tecac'].latitude, 
                            means[var]['tecac'], zorder=1,
                            cmap=cmaps[var], norm=norms[var], transform=ccrs.PlateCarree())

    cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var], norm=norms[var]),
                            ax=axICE6G[var],
                            orientation='vertical', shrink=0.8, pad=0.05)
    
    axGLAC1D[var].pcolormesh(means[var]['xoupa'].longitude, means[var]['xoupa'].latitude, 
                             means[var]['xoupa'], zorder=1,
                             cmap=cmaps[var], norm=norms[var], transform=ccrs.PlateCarree())

    cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var], norm=norms[var]),
                            ax=axGLAC1D[var],
                            orientation='vertical', shrink=0.8, pad=0.05)
    
    if var == 'ice':
        temp = np.nan_to_num(means[var]['tecac']) - np.nan_to_num(means[var]['xoupa'])
        axDIFF[var].pcolormesh(means[var]['xoupa'].longitude, means[var]['xoupa'].latitude, 
                               np.where(temp==0, np.nan, temp), zorder=1,
                               cmap=cmaps[var+'_diff'], norm=norms[var+'_diff'], transform=ccrs.PlateCarree())

    else:
        axDIFF[var].pcolormesh(means[var]['xoupa'].longitude, means[var]['xoupa'].latitude, 
                               means[var]['tecac'] - means[var]['xoupa'], zorder=1,
                               cmap=cmaps[var+'_diff'], norm=norms[var+'_diff'], transform=ccrs.PlateCarree())
    cbar = fig.colorbar(mappable=matplotlib.cm.ScalarMappable(cmap=cmaps[var+'_diff'], norm=norms[var+'_diff']),
                           ax=axDIFF[var],
                           orientation='vertical', shrink=0.8, pad=0.05)
    cbar.set_label(labels_cbar[var], size='large')
    cbar.ax.tick_params(labelsize='medium')


# Plot sea ice
ls = {'wice':'-', 'sice':'--'}
for var in ['wice', 'sice']:
    axICE6G['ice'].contour(means[var]['tecac'].longitude, means[var]['tecac'].latitude, 
                         means[var]['tecac'], levels=[50], linestyles=ls[var],
                         colors=color_expt['xpfjb'], transform=ccrs.PlateCarree())

    axGLAC1D['ice'].contour(means[var]['xoupa'].longitude, means[var]['xoupa'].latitude, 
                         means[var]['xoupa'], levels=[50],linestyles=ls[var],
                         colors=color_expt['xpfjd'], transform=ccrs.PlateCarree())

    axDIFF['ice'].contour(means[var]['tecac'].longitude, means[var]['tecac'].latitude, 
                         means[var]['tecac'], levels=[50],linestyles=ls[var],
                         colors=color_expt['xpfjb'], transform=ccrs.PlateCarree())
    axDIFF['ice'].contour(means[var]['xoupa'].longitude, means[var]['xoupa'].latitude, 
                         means[var]['xoupa'], levels=[50],linestyles=ls[var],
                         colors=color_expt['xpfjd'], transform=ccrs.PlateCarree())

# Plot winds
axICE6G['ws'].quiver(means['u']['tecac'].longitude[::2], means['v']['tecac'].latitude[::2], 
                     means['u']['tecac'].values[::2,::2], means['v']['tecac'].values[::2,::2],
                     scale=100, transform=ccrs.PlateCarree())
    
axGLAC1D['ws'].quiver(means['u']['xoupa'].longitude[::2], means['v']['xoupa'].latitude[::2], 
                      means['u']['xoupa'].values[::2,::2], means['v']['xoupa'].values[::2,::2],
                      scale=100, transform=ccrs.PlateCarree())

axDIFF['ws'].quiver(means['u']['tecac'].longitude[::2], means['v']['tecac'].latitude[::2], 
                    means['u']['tecac'].values[::2,::2] - means['u']['xoupa'].values[::2,::2],
                    means['v']['tecac'].values[::2,::2] - means['v']['xoupa'].values[::2,::2],
                    scale=100, transform=ccrs.PlateCarree())

# Plot land-sea masks    
for ax in axICE6G.values():
    ax.pcolormesh(lsm['ICE6G'].longitude, lsm['ICE6G'].latitude,
                  xr.where(lsm['ICE6G']==0, np.nan, lsm['ICE6G']),
                  zorder=0,
                  transform=ccrs.PlateCarree(), cmap=ListedColormap(['xkcd:pale brown']))

for ax in axGLAC1D.values():
    ax.pcolormesh(lsm['GLAC-1D'].longitude, lsm['GLAC-1D'].latitude,
                  xr.where(lsm['GLAC-1D']==0, np.nan, lsm['GLAC-1D']),
                  zorder=0,
                  transform=ccrs.PlateCarree(), cmap=ListedColormap(['xkcd:pale brown']))

for ax in axDIFF.values():
    ax.pcolormesh(lsm['GLAC-1D'].longitude, lsm['GLAC-1D'].latitude,
                  xr.where(lsm['GLAC-1D']+lsm['ICE6G']==0, np.nan, lsm['GLAC-1D']+lsm['ICE6G']),
                  zorder=0, 
                  transform=ccrs.PlateCarree(), cmap=ListedColormap(['xkcd:pale brown']))

for ax in [axICE6G, axGLAC1D, axDIFF]: ax['ws'].coastlines(lw=0.5)

# -----

# High and low pressure systems
for system in ['arc', 'icd', 'az']:
    for ax in axICE6G, axDIFF:
        ax['ice'].scatter(pressure_systems[system]['tecac'][0], pressure_systems[system]['tecac'][1],
                                marker=marker_pressure[system], c=color_expt['xpfjb'], edgecolors='white', linewidths=0.4,
                                transform=ccrs.PlateCarree())
    for ax in axGLAC1D, axDIFF:
        ax['ice'].scatter(pressure_systems[system]['xoupa'][0], pressure_systems[system]['tecac'][1],
                                marker=marker_pressure[system], c=color_expt['xpfjd'], edgecolors='white', linewidths=0.4,
                                 transform=ccrs.PlateCarree())
    axICE6G['ice'].scatter([],[],marker=marker_pressure[system], c='black', edgecolors='white', linewidths=0.4,
                            transform=ccrs.PlateCarree(), label=label_pressure[system])
    
# Deep water formation sites
for zone in ['nor', 'icd', 'irm', 'ls']:
    for ax in axICE6G, axGLAC1D:
        for var in ['mld']:
            ax[var].fill(dwfs[zone][0], dwfs[zone][1], 
                       color=color_zones[zone], linestyle = "-", alpha = 1, linewidth=1, 
                       transform=ccrs.PlateCarree(), fill=False)
            ax[var].fill(dwfs[zone][0], dwfs[zone][1], 
                       color=color_zones[zone], linestyle = "-", alpha = 0.2, 
                       transform=ccrs.PlateCarree(), fill=True, label=label_zones[zone])

# Legends
axICE6G['ice'].legend(loc='upper left', bbox_to_anchor=(-0.5, 0.16), fontsize='medium', frameon=False)
axICE6G['mld'].legend(loc='upper left', bbox_to_anchor=(-0.5, 0.19), fontsize='medium', frameon=False)
    
# ------

# Experiment Box
def plot_experiment_box(name, ax, color, size='x-large'):
    ax_box = ax.get_position()
    
    cax = fig.add_axes([ax_box.x0, ax_box.y1+0.02,
                    ax_box.width+0.02, 
                    ax_box.height/4])
    
    cax.get_xaxis().set_visible(False)
    cax.get_yaxis().set_visible(False)
    for loc in ['left', 'right', 'bottom', 'top']: cax.spines[loc].set_visible(False)
    cax.set_facecolor(color)

    at = AnchoredText(name, loc=10, frameon=False,
                        prop=dict(backgroundcolor=color,
                                  size=size, color='white'))
    
    cax.add_artist(at)
    
plot_experiment_box(name_expt['tecac'], axICE6G['ice'], color_expt['tecac'])
plot_experiment_box(name_expt['xoupa'], axGLAC1D['ice'], color_expt['xoupa'])
plot_experiment_box(name_expt['tecac'] + ' - ' + name_expt['xoupa'], axDIFF['ice'], 'darkslategrey', size='large')

# Annotations
# axICE6G['ice'].set_title(name_expt['tecac'], size='x-large', color=color_expt['tecac'])
# axGLAC1D['ice'].set_title(name_expt['xoupa'], size='x-large',  color=color_expt['xoupa'])
# axDIFF['ice'].set_title(name_expt['tecac'] + ' - ' + name_expt['xoupa'], size='x-large', color='k')

# axICE6G['mld'].annotate('Winter mixed layer depth\n($m$)', xy=(-0.3, 0.5), xycoords='axes fraction',
#                          ha='center', va='center', rotation=90,  size='large')
# axICE6G['ice'].annotate('Ice sheet elevation\n($m$)', xy=(-0.3, 0.5), xycoords='axes fraction',
#                          ha='center', va='center', rotation=90,  size='large')
# axICE6G['ws'].annotate('10m Wind stress\n($m.s^{-1}$)', xy=(-0.3, 0.5), xycoords='axes fraction',
#                          ha='center', va='center', rotation=90,  size='large')

i = count(0)
for var in ['ice', 'mld', 'ws']:
    for ax in axICE6G, axGLAC1D, axDIFF:
        txt = ax[var].annotate(alc[next(i)], xy=(-0.02,0.87), xycoords='axes fraction', size=14, weight='bold')


In [ ]:
print(means['mld']['tecac'].sel(latitude=slice(35,60)).max().values)
print((means['mld']['tecac'] * masks_na['eur']).max().values)
print((means['mld']['xoupa'] * masks_na['irm']).mean().values)
print((means['mld']['xoupa'] * masks_na['irm']).max().values)
print((means['ws']['tecac'] - means['ws']['xoupa'].sel(latitude=slice(60,35))).max().values)
print((means['ws']['tecac'] - means['ws']['xoupa'].sel(latitude=slice(60,35))).min().values)

In [ ]:
# fig.savefig(f"{output_folder}/control.png", bbox_extra_artists=(), format='png')

## Figure 3 - Dynamics of the transient simulations

*TO DO*: Check N2 in XQCTC

In [ ]:
filt = util.ButterLowPass(order=1, fc=2*10**-3, fs=1, mult=2)

In [ ]:
amoc = {}

for expt in database.keys():
# for expt in ['xoupa', 'xqctb', 'xqctc']:
    print(f"Loading {expt}...")
    amoc[expt] = xr.open_dataset(
        f"{database[expt]}/time_series/{expt}.merid.annual.nc").Merid_Atlantic.sel(
        latitude=26.5, method='nearest').max('depth')

In [ ]:
mld_zone, mldf_zone = {}, {}

for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'xqctc']: 
    print(f"Loading {expt} MLD...")
    mld_zone[expt], mldf_zone[expt] = {}, {}
    temp_mld = xr.open_dataset(
                f"{database[expt]}/na_variables/{expt}.mld.zone_na.annual.nc").mld
    
    for zone in masks_na.keys():
        mld_zone[expt][zone] = temp_mld.sel(zone=zone)[1:]
        mldf_zone[expt][zone] = filt.process(mld_zone[expt][zone].values)

In [ ]:
mld_control = {}
for expt in ['xoupa', 'tecac']: 
    mld_control[expt] = {}
    temp_mld = xr.open_dataset(
                f"{database[expt]}/na_variables/{expt}.mld.zone_na.annual.nc").mld
    
    for zone in masks_na.keys():
        mld_control[expt][zone] = temp_mld.sel(zone=zone).mean('t')

In [ ]:
N2 = {}

for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'xqctc']: 
    ocnd = xr.open_dataset(f"{database[expt]}/na_variables/{expt}.oceandenspg.zone_na.annual.nc").density

    N2[expt] = {}

    for zone in masks_na.keys():
        N2[expt][zone] = (-9.81/1025*ocnd.sel(zone=zone).sel(
            depth_1=slice(0,1000)).diff(dim='depth_1', label='lower')\
                    /ocnd.sel(zone=zone).sel(
            depth_1=slice(0,1000)).depth_1.diff(dim='depth_1', label='lower')).sum('depth_1')

In [ ]:
eml_amoc = {}
for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'GLAC-1D', 'ICE6G']:
    eml_amoc[expt] = xr.open_dataset(f"{data_folder}/mw_bins/eml_amoc.{expt}.nc").Merid_Atlantic.sortby('year')
    
eml_mw = {}
for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'GLAC-1D', 'ICE6G']:
    eml_mw[expt] = xr.open_dataset(f"{data_folder}/mw_bins/eml_mw_roll.{expt}.nc").sortby('year')
    

In [ ]:
lreg = {'mw':{},'GLAC-1D':{}, 'ICE6G':{}}
indexes = {'GLAC-1D':{}, 'ICE6G':{}}
x = {'GLAC-1D':{}, 'ICE6G':{}}


# GLAC-1D
indexes['GLAC-1D']['warm'] = np.where(eml_mw['GLAC-1D'].tot<0.05)[0]
lreg['GLAC-1D']['warm'] = stats.linregress(eml_mw['GLAC-1D']['tot'].isel(year=indexes['GLAC-1D']['warm']), 
                                           eml_amoc['GLAC-1D'].isel(year=indexes['GLAC-1D']['warm']))
x['GLAC-1D']['warm'] = np.linspace(eml_mw['GLAC-1D']['tot'].min().values[()], 0.05, 10) 

indexes['GLAC-1D']['cold'] = np.where(eml_mw['GLAC-1D'].tot>0.15)[0]
lreg['GLAC-1D']['cold'] = stats.linregress(eml_mw['GLAC-1D']['tot'].isel(year=indexes['GLAC-1D']['cold']), 
                                           eml_amoc['GLAC-1D'].isel(year=indexes['GLAC-1D']['cold']))
x['GLAC-1D']['cold']  = np.linspace(0.15, eml_mw['GLAC-1D']['tot'].max().values[()], 10) 


# ICE6G
indexes['ICE6G']['warm'] = np.where(eml_mw['ICE6G'].tot<0.12)[0]
lreg['ICE6G']['warm'] = stats.linregress(eml_mw['ICE6G']['tot'].isel(year=indexes['ICE6G']['warm']), 
                                           eml_amoc['ICE6G'].isel(year=indexes['ICE6G']['warm']))
x['ICE6G']['warm'] = np.linspace(eml_mw['ICE6G']['tot'].min().values[()], 0.12, 10) 

indexes['ICE6G']['cold'] = np.where(eml_mw['ICE6G'].tot>0.12)[0]
lreg['ICE6G']['cold'] = stats.linregress(eml_mw['ICE6G']['tot'].isel(year=indexes['ICE6G']['cold']), 
                                           eml_amoc['ICE6G'].isel(year=indexes['ICE6G']['cold']))
x['ICE6G']['cold']  = np.linspace(0.12, eml_mw['ICE6G']['tot'].max().values[()], 10) 


for zone in ['arc', 'gin', 'elwg']:
    lreg['mw'][zone] = stats.linregress(eml_mw['GLAC-1D']['tot'].dropna(dim='year'), 
                                        eml_mw['GLAC-1D'][zone].dropna(dim='year'))

x['mw'] = np.linspace(eml_mw['GLAC-1D']['tot'].min().values[()], eml_mw['GLAC-1D']['tot'].max().values[()], 10) 


In [ ]:
salinity_clusters = {}

for expt in ['xpfjb', 'xpfjc', 'xpfjd', 'xpfje', 'xqctc']: 
    salinity_clusters[expt] = xr.open_dataset(
        f"{database[expt]}/salinity_budgets/{expt}.salinity_clusters.decadal.nc").salinity_cluster

In [ ]:
def plot_LineCollection(ax, t, ts, alpha):
    points = np.array([t, ts]).T.reshape(-1,1,2)
    segments = np.concatenate([points[:-1],points[1:]], axis=1)
    lc = LineCollection(segments, cmap=scm.cm.roma, linewidth=1, alpha=alpha)
    lc.set_array(np.arange(len(ts)))

    ax.add_collection(lc)

def plot_experiment_box(name, ax, color, size='x-large'):
    ax_box = ax.get_position()
    
    cax = fig.add_axes([ax_box.x0, ax_box.y1-0.01,
                    ax_box.width, 
                    ax_box.height+0.01])
    
    cax.get_xaxis().set_visible(False)
    cax.get_yaxis().set_visible(False)
    for loc in ['left', 'right', 'bottom', 'top']: cax.spines[loc].set_visible(False)
    cax.set_facecolor(color)

    at = AnchoredText(name, loc=10, frameon=False,
                        prop=dict(backgroundcolor=color,
                                  size=size, color='white'))
    
    cax.add_artist(at)

# ------

fig = plt.figure(figsize=(12, 18), dpi=300)

grid = fig.add_gridspec(4, 3, hspace=0.1, wspace=0.2, height_ratios=[1,18,2,12])

axEXPT, axAMOC, axN2, axS, axMLD = {}, {}, {}, {}, {}
axEML = {}

iterator = count(0)
experiments = ['xpfjd', 'xpfjc', 'xqctc']

# Grids
for expt in experiments:
    i = next(iterator)

    axEXPT[expt] = fig.add_subplot(grid[0,i])

    subgrid = gridspec.GridSpecFromSubplotSpec(4, 1, subplot_spec=grid[1,i], hspace=0, height_ratios=[6,5,5,8])
    
    axAMOC[expt] = fig.add_subplot(subgrid[0], facecolor='None')
    axN2[expt] = fig.add_subplot(subgrid[1], sharex=axAMOC[expt], facecolor='None')
    axS[expt] = fig.add_subplot(subgrid[2], sharex=axAMOC[expt], facecolor='None')    
    axMLD[expt] = fig.add_subplot(subgrid[3], facecolor='None')

    # Experiment Box
    plot_experiment_box(name_expt[expt], axEXPT[expt], color_expt[expt])
    axEXPT[expt].set_visible(False)

    
axTS = np.array([list(axAMOC.values()), list(axN2.values()), list(axS.values())]).flat

subgrid = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=grid[3,:], hspace=0, height_ratios=[1,8])

axEXPT['GLAC-1D'] = fig.add_subplot(subgrid[0,0])
plot_experiment_box("GLAC-1D simulations", axEXPT['GLAC-1D'], color_expt['xoupa'])
axEXPT['GLAC-1D'].set_visible(False)
axEML['GLAC-1D'] = fig.add_subplot(subgrid[1,0], facecolor='None')

axEXPT['ICE6G'] = fig.add_subplot(subgrid[0,1])
plot_experiment_box("ICE6G simulations", axEXPT['ICE6G'], color_expt['tecac'])
axEXPT['ICE6G'].set_visible(False)
axEML['ICE6G'] = fig.add_subplot(subgrid[1,1], sharex=axEML['GLAC-1D'], sharey=axEML['GLAC-1D'], facecolor='None')


# -------

# AMOC
for expt in experiments:
    plot_LineCollection(axAMOC[expt], amoc[expt].t.dt.year - start_expt[expt], amoc[expt], 0.03)
    plot_LineCollection(axAMOC[expt], amoc[expt].t.dt.year - start_expt[expt], util.rmean(amoc[expt],30) , 1)
   
# Plot N2 index
for expt in experiments:
    for zone in ['gin', 'irm', 'eur']:
        axN2[expt].plot(N2[expt][zone].t.dt.year - start_expt[expt], util.rmean(N2[expt][zone],30)*1e4,
                  color=color_zones[zone], label=label_zones[zone])

# Plot Salinity budgets
for expt in experiments:
    for zone in ['na', 'tpa', 'pac', 'arc']:
        salinity_anomaly = (salinity_clusters[expt].sel(zone=zone).sel(depth='tot') - salinity_clusters[expt].sel(zone=zone).sel(depth='tot')[0])
        salinity_anomaly -= util.rmean((salinity_clusters[expt].sel(zone=zone).sel(depth='tot') - salinity_clusters[expt].sel(zone=zone).sel(depth='tot')[0]), n=100)
        axS[expt].plot(salinity_clusters[expt].sel(zone=zone).sel(depth='tot').t.dt.year \
                            - start_expt[expt],
                   salinity_anomaly*1e-15, 
                   color=color_zones[zone], label=label_zones[zone])
    
# Phases
for expt in experiments:
    axMLD[expt].scatter(mldf_zone[expt]['gin'], mldf_zone[expt]['irm'], 
                               c =scm.cm.roma(np.linspace(0, 1, len(mldf_zone[expt]['gin']))),
                               marker='+', alpha=0.1, zorder=2)

# Control points
for expt in ['xpfjd']:
    axMLD[expt].scatter(mld_control['xoupa']['gin'], mld_control['xoupa']['irm'],
                        c=color_expt['xoupa'],
                         marker='.', s=10, edgecolors='None', alpha=0.4, linestyle='None', zorder=2)
    
for expt in ['xpfjc']:
    axMLD[expt].scatter(mld_control['tecac']['gin'], mld_control['tecac']['irm'],
                        c=color_expt['tecac'],
                         marker='.', s=10, edgecolors='None', alpha=0.4, linestyle='None', zorder=2)

# -----

# General parameters
for ax in axTS:
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(which='both', color='lightgrey', linestyle='--', linewidth=0.2, zorder=0)
    ax.tick_params(axis='both', which='minor', size=3)

# Paramters time series
for expt in experiments:
    axAMOC[expt].xaxis.set(ticks_position='top', label_position='top')
    axAMOC[expt].set_xlim([-21500,-13000])
    axAMOC[expt].set_ylim([0,25])
    axAMOC[expt].set_xlabel('Years')
    for loc in ['right', 'bottom']: axAMOC[expt].spines[loc].set_visible(False)
        
    axN2[expt].tick_params(axis='x', colors='None', which='both')
    axN2[expt].set_ylim([1,-25])
    axN2[expt].yaxis.set(ticks_position='right', label_position='right')
    for loc in ['left', 'top', 'bottom']: axN2[expt].spines[loc].set_visible(False)
    
    axS[expt].tick_params(axis='x', colors='None', which='both')
    axS[expt].axhline(0, color='black', lw=1, linestyle='--')
    # axS[expt].set_ylim([-400,50])
    for loc in ['right', 'top', 'bottom']: axS[expt].spines[loc].set_visible(False)
    
axAMOC['xpfjd'].set_ylabel('AMOC index\n($Sv$)')
axN2['xqctc'].set_ylabel(r"$N^2$ Stratification index""\n"r"($e^{-4}s^{-2}$)")
axS['xpfjd'].set_ylabel('Salt content - 100-year running mean\n($e15 kg$)')
axS['xpfjc'].legend(fontsize='x-small', frameon=False, bbox_to_anchor=(0.02, 1.05), loc='upper left')
axN2['xpfjd'].legend(fontsize='small', frameon=False)

# Paramters Phase
for expt in experiments:
    axMLD[expt].set_xlim([10,110])
    axMLD[expt].set_ylim([20,110])
    axMLD[expt].set_xlabel('Mixed layer depth\nin the GIN seas ($m$)')
    axMLD[expt].grid(which='both', color='lightgrey', linestyle='--', linewidth=0.2, zorder=0)

axMLD['xqctc'].set_ylabel('Mixed layer depth\nin the Irminger Sea ($m$)')
axMLD['xqctc'].yaxis.set(ticks_position='right', label_position='right')

# -----

def _plot_expts(ax, expts):
    for expt in expts:
        ax.plot(eml_mw[expt]['tot'], util.rmean(eml_amoc[expt],30), color=color_expt[expt], alpha=1, lw=1)
        ax.scatter(eml_mw[expt]['tot'], eml_amoc[expt], 
                color=color_expt[expt], marker='.', s=10, edgecolors='None', alpha=0.2, linestyle='None')

# Plot experiments states
_plot_expts(axEML['GLAC-1D'], ['xpfjd', 'xpfje'])
_plot_expts(axEML['ICE6G'], ['xpfjb', 'xpfjc'])

# Plot fits
for mw in ['GLAC-1D', 'ICE6G']:
    for mode in ['warm', 'cold']:
        axEML[mw].plot(x[mw][mode],
                       lreg[mw][mode].slope * x[mw][mode] + lreg[mw][mode].intercept, 
                       color=color_spans['merid'] if mode=='warm' else color_spans['cold'],
                       ls='--', label=f"{mode} fit ($r^2$={lreg[mw][mode].rvalue**2:.2f})")

# -----

# Annotate fits
axEML['GLAC-1D'].set_xlim([0,0.35])
axEML['GLAC-1D'].set_ylim([0,26])
axEML['GLAC-1D'].axvspan(0, 0.05, alpha=0.05, color=color_spans['merid'], linewidth=0, hatch='//', edgecolor=color_spans['merid'])
axEML['GLAC-1D'].axvspan(0.05, 0.15, alpha=0.3, color=color_spans['zonal'], linewidth=0)
axEML['GLAC-1D'].axvspan(0.15, 0.35, alpha=0.05, color=color_spans['cold'], linewidth=0, hatch='//', edgecolor=color_spans['cold'])
xlim = axEML['GLAC-1D'].get_xlim()
x_frac = (0.1 - xlim[0]) / (xlim[1] - xlim[0])
axEML['GLAC-1D'].annotate("Window of opportunity", xy=(x_frac, 1.02), xycoords='axes fraction',
                          ha='center', va='bottom', fontsize='small', color=color_spans['zonal'])
axEML['ICE6G'].axvspan(0, 0.12, alpha=0.05, color=color_spans['merid'], linewidth=0, hatch='//', edgecolor=color_spans['merid'])
axEML['ICE6G'].axvspan(0.12, 0.35, alpha=0.05, color=color_spans['cold'], linewidth=0, hatch='//', edgecolor=color_spans['cold'])

# Parameters for mw panel
axEML['GLAC-1D'].set_xlabel("Total discharge (Sv)", fontsize='large')
axEML['GLAC-1D'].set_ylabel("AMOC index (Sv)", fontsize='large')
for loc in ['right', 'top']:
    axEML['GLAC-1D'].spines[loc].set_visible(False)

axEML['ICE6G'].set_xlabel("Total discharge (Sv)", fontsize='large')
axEML['ICE6G'].yaxis.set_label_position('right')
axEML['ICE6G'].yaxis.set_ticks_position('right')
for loc in ['left', 'top']:
    axEML['ICE6G'].spines[loc].set_visible(False)

# Annotations
i = count(0)
for expt in experiments:
    txt = axAMOC[expt].annotate(alc[next(i)], xy=(0.93,0.04), xycoords='axes fraction', size=12, weight='bold')
for expt in experiments:
    txt = axN2[expt].annotate(alc[next(i)], xy=(0.93,0.84), xycoords='axes fraction', size=12, weight='bold')
for expt in experiments:
    txt = axS[expt].annotate(alc[next(i)], xy=(0.93,0.84), xycoords='axes fraction', size=12, weight='bold')
for expt in experiments:
    txt = axMLD[expt].annotate(alc[next(i)], xy=(0.93,0.04), xycoords='axes fraction', size=12, weight='bold')
axEML['GLAC-1D'].annotate(alc[next(i)], xy=(0.93,0.94), xycoords='axes fraction', size=12, weight='bold')
axEML['ICE6G'].annotate(alc[next(i)], xy=(0.93,0.94), xycoords='axes fraction', size=12, weight='bold')


In [ ]:
fig.savefig(f"{output_folder}/dynamics.png", bbox_extra_artists=(), bbox_inches='tight', format='png')